# Q Colab Free Training

This notebook is bound to the tracked Q + Immaculate hybrid session `q-hybrid-harbor-opt-2384cf5-bench-v12`.

It does two real things:
- rebuilds the Immaculate orchestration bundle from the same stamped session inputs
- runs a bounded Q micro-train when Colab exposes enough GPU memory

Truth boundary:
- this does **not** replace the heavier HF Jobs or OCI lane
- this is a free supplemental lane for session doctoring, bundle replay, and bounded Q updates
- if the available Colab GPU is too small, the notebook stays truthful and stops at doctor plus dry-run

In [ ]:
SESSION_ID = "q-hybrid-harbor-opt-2384cf5-bench-v12"
REPO_SLUG = "PossumXI/Immaculate"
REPO_URL = "https://github.com/PossumXI/Immaculate.git"
REPO_COMMIT = "848d44f8dbe409a8227bab61f88eccbf078dd056"
HF_DATASET_REPO = "TruLumecreator/immaculate-q-cloud-bundles"
HF_ARCHIVE_PATH = "sessions/q-hybrid-harbor-opt-2384cf5-bench-v12/q-hybrid-harbor-opt-2384cf5-bench-v12-cloud-bundle.tar.gz"
HF_MANIFEST_PATH = "sessions/q-hybrid-harbor-opt-2384cf5-bench-v12/bundle-manifest.json"
SESSION_MANIFEST_RELATIVE = ".training-output/q/latest-hybrid-session.json"
Q_CONFIG_RELATIVE = ".training-output/q/q-lora-config-harbor-opt-2384cf5-bench-v12.json"
IMMACULATE_BUNDLE_OUTPUT = ".training-output/immaculate/immaculate-training-bundle-q-hybrid-harbor-opt-2384cf5-bench-v12.json"
MICRO_CONFIG_RELATIVE = ".training-output/q/colab/q-colab-micro-config.json"
MICRO_MAX_STEPS = 24
MICRO_MAX_SEQ_LENGTH = 2048
MIN_GPU_MEMORY_GB = 20
RUN_MICRO_TRAIN = True
MOUNT_DRIVE = True

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
from getpass import getpass
from pathlib import Path

def run(command, cwd=None, env=None):
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=cwd, env=env, check=True)
    return completed.returncode

def safe_extract(archive_path: Path, target_dir: Path):
    target_dir = target_dir.resolve()
    with tarfile.open(archive_path, "r:gz") as handle:
        for member in handle.getmembers():
            candidate = (target_dir / member.name).resolve()
            if not str(candidate).startswith(str(target_dir)):
                raise ValueError(f"Unsafe tar entry: {member.name}")
        handle.extractall(target_dir)

if "google.colab" in sys.modules and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content/immaculate-colab")
REPO_ROOT = WORKSPACE / "repo"
WORKSPACE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

run(["git", "clone", REPO_URL, str(REPO_ROOT)])
run(["git", "checkout", REPO_COMMIT], cwd=str(REPO_ROOT))

In [ ]:
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "huggingface_hub",
        "datasets",
        "transformers",
        "trl",
        "accelerate",
        "peft",
        "bitsandbytes",
        "wandb",
        "unsloth",
    ]
)

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("HF_TOKEN: ").strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required to pull the staged cloud bundle.")

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "").strip()

In [ ]:
from huggingface_hub import hf_hub_download

archive_path = Path(
    hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=HF_ARCHIVE_PATH,
        repo_type="dataset",
        token=HF_TOKEN,
    )
)
manifest_path = Path(
    hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=HF_MANIFEST_PATH,
        repo_type="dataset",
        token=HF_TOKEN,
    )
)

safe_extract(archive_path, REPO_ROOT)
print("Archive:", archive_path)
print("Manifest:", manifest_path)

In [ ]:
run(
    [
        sys.executable,
        "training/q/run_q_training_session.py",
        "--doctor",
        "--session",
        SESSION_MANIFEST_RELATIVE,
    ],
    cwd=str(REPO_ROOT),
)

run(
    [
        sys.executable,
        "training/immaculate/build_immaculate_training_bundle.py",
        "--output",
        IMMACULATE_BUNDLE_OUTPUT,
    ],
    cwd=str(REPO_ROOT),
)

In [ ]:
colab_output_root = Path("/content/immaculate-colab-output") / SESSION_ID
if Path("/content/drive").exists():
    drive_output_root = Path("/content/drive/MyDrive/immaculate/q-runs") / SESSION_ID
    drive_output_root.parent.mkdir(parents=True, exist_ok=True)
    colab_output_root = drive_output_root
colab_output_root.mkdir(parents=True, exist_ok=True)
micro_output_dir = colab_output_root / "q-colab-free"

run(
    [
        sys.executable,
        "training/q/build_q_colab_micro_config.py",
        "--config",
        Q_CONFIG_RELATIVE,
        "--output",
        MICRO_CONFIG_RELATIVE,
        "--max-steps",
        str(MICRO_MAX_STEPS),
        "--max-seq-length",
        str(MICRO_MAX_SEQ_LENGTH),
        "--gradient-accumulation-steps",
        "8",
        "--output-dir",
        str(micro_output_dir),
    ] + (["--disable-wandb"] if not WANDB_API_KEY else []),
    cwd=str(REPO_ROOT),
)

run(
    [
        sys.executable,
        "training/q/train_q_lora_unsloth.py",
        "--config",
        MICRO_CONFIG_RELATIVE,
        "--session-manifest",
        SESSION_MANIFEST_RELATIVE,
        "--dry-run",
    ],
    cwd=str(REPO_ROOT),
)

In [ ]:
import torch

gpu_ready = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_ready else "cpu-only"
gpu_memory_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1)
    if gpu_ready
    else 0.0
)
print({"gpu_ready": gpu_ready, "gpu_name": gpu_name, "gpu_memory_gb": gpu_memory_gb})

if RUN_MICRO_TRAIN and gpu_ready and gpu_memory_gb >= MIN_GPU_MEMORY_GB:
    launch_env = dict(os.environ)
    launch_env["HF_TOKEN"] = HF_TOKEN
    if WANDB_API_KEY:
        launch_env["WANDB_API_KEY"] = WANDB_API_KEY
    if WANDB_ENTITY:
        launch_env["WANDB_ENTITY"] = WANDB_ENTITY
    if WANDB_PROJECT:
        launch_env["WANDB_PROJECT"] = WANDB_PROJECT
    run(
        [
            sys.executable,
            "training/q/train_q_lora_unsloth.py",
            "--config",
            MICRO_CONFIG_RELATIVE,
            "--session-manifest",
            SESSION_MANIFEST_RELATIVE,
        ],
        cwd=str(REPO_ROOT),
        env=launch_env,
    )
else:
    print(
        "Skipping bounded Q micro-train because the current Colab runtime is too small. "
        "Doctor, bundle replay, and dry-run validation still completed."
    )